# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.

In [1]:
import sys
print(sys.executable)

/home/sam/ai/projects/tinyml-arduino/bin/python


In [2]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

2026-05-20 23:56:48.597709: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-20 23:56:48.599317: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-20 23:56:48.619560: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-20 23:56:48.619594: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-20 23:56:48.619643: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to regi

TensorFlow version: 2.14.1
TF-MOT version: 0.8.0


---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset provided in `wine.zip`. The zip file is extracted and `wine.data` is loaded directly.

In [5]:
# Load the Wine dataset from wine.zip.
import zipfile

# Extract wine.zip and load wine.data
with zipfile.ZipFile("wine.zip", "r") as z:
    z.extractall("wine_data")

feature_names = [
    "Alcohol", "Malic_acid", "Ash", "Alcalinity_of_ash", "Magnesium",
    "Total_phenols", "Flavanoids", "Nonflavanoid_phenols", "Proanthocyanins",
    "Color_intensity", "Hue", "OD280_OD315", "Proline"
]

df = pd.read_csv("wine_data/wine.data", header=None,
                 names=["Class"] + feature_names)

# Convert class labels from 1-based to 0-based
df["Class"] = df["Class"] - 1

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)

Number of classes: 3
Number of features: 13

Feature statistics:
                          min      max        mean         std
Alcohol                11.03    14.83   13.000618    0.811827
Malic_acid              0.74     5.80    2.336348    1.117146
Ash                     1.36     3.23    2.366517    0.274344
Alcalinity_of_ash      10.60    30.00   19.494944    3.339564
Magnesium              70.00   162.00   99.741573   14.282484
Total_phenols           0.98     3.88    2.295112    0.625851
Flavanoids              0.34     5.08    2.029270    0.998859
Nonflavanoid_phenols    0.13     0.66    0.361854    0.124453
Proanthocyanins         0.41     3.58    1.590899    0.572359
Color_intensity         1.28    13.00    5.058090    2.318286
Hue                     0.48     1.71    0.957449    0.228572
OD280_OD315             1.27     4.00    2.611685    0.709990
Proline               278.00  1680.00  746.893258  314.907474

Class distribution:
 Class
0    59
1    71
2    48
Name: count, d

## Problem 1 - Part (a)
### Base Model Training and Evaluation

In [6]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

X = df[feature_names].values
y = df["Class"].values

print("Feature matrix shape:", X.shape)
print("Label vector shape:  ", y.shape)

Feature matrix shape: (178, 13)
Label vector shape:   (178,)


In [7]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print("X_train shape:", X_train.shape)
print("X_test  shape:", X_test.shape)

X_train shape: (124, 13)
X_test  shape: (54, 13)


In [8]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [9]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat  = to_categorical(y_test,  num_classes=num_classes)

print("y_train_cat shape:", y_train_cat.shape)
print("y_test_cat  shape:", y_test_cat.shape)

y_train_cat shape: (124, 3)
y_test_cat  shape: (54, 3)


In [10]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

model = Sequential([
    Dense(64, activation='relu', input_shape=(num_features,)),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                896       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 3)                 99        
                                                                 
Total params: 3075 (12.01 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


2026-05-20 23:56:49.692710: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:880] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-05-20 23:56:49.693029: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2211] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [11]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train_scaled, y_train_cat,
    epochs=20,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

train_acc = history.history['accuracy'][-1]
print(f"\nFinal Training Accuracy: {train_acc:.4f}")

Epoch 1/20
13/13 [==============================] - 0s 9ms/step - loss: 1.1179 - accuracy: 0.4545 - val_loss: 0.9019 - val_accuracy: 0.5600
Epoch 2/20
13/13 [==============================] - 0s 2ms/step - loss: 0.7802 - accuracy: 0.6768 - val_loss: 0.6383 - val_accuracy: 0.7600
Epoch 3/20
13/13 [==============================] - 0s 2ms/step - loss: 0.5672 - accuracy: 0.8687 - val_loss: 0.4586 - val_accuracy: 0.8800
Epoch 4/20
13/13 [==============================] - 0s 2ms/step - loss: 0.4055 - accuracy: 0.9596 - val_loss: 0.3385 - val_accuracy: 0.9600
Epoch 5/20
13/13 [==============================] - 0s 2ms/step - loss: 0.2879 - accuracy: 0.9899 - val_loss: 0.2611 - val_accuracy: 0.9600
Epoch 6/20
13/13 [==============================] - 0s 2ms/step - loss: 0.2114 - accuracy: 0.9899 - val_loss: 0.2048 - val_accuracy: 0.9600
Epoch 7/20
13/13 [==============================] - 0s 2ms/step - loss: 0.1582 - accuracy: 0.9899 - val_loss: 0.1713 - val_accuracy: 0.9600
Epoch 8/20
13/13 [==

In [12]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

loss, test_acc = model.evaluate(X_test_scaled, y_test_cat, verbose=0)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss:     {loss:.4f}")

y_pred_probs = model.predict(X_test_scaled)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test_cat,   axis=1)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=['Class 0', 'Class 1', 'Class 2']))

print("Confusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Test Accuracy: 1.0000
Test Loss:     0.0572
2/2 [==============================] - 0s 2ms/step

Classification Report:
              precision    recall  f1-score   support

     Class 0       1.00      1.00      1.00        19
     Class 1       1.00      1.00      1.00        21
     Class 2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


In [13]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

def file_size_kb(filename):
    """Return the size of a file in kilobytes."""
    return os.path.getsize(filename) / 1024

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_base = converter.convert()

with open('model_base.tflite', 'wb') as f:
    f.write(tflite_base)

print(f"Base (float32) TFLite model size: {file_size_kb('model_base.tflite'):.2f} KB")

INFO:tensorflow:Assets written to: /tmp/tmpgeg4ll1i/assets


INFO:tensorflow:Assets written to: /tmp/tmpgeg4ll1i/assets


Base (float32) TFLite model size: 14.07 KB


2026-05-20 23:56:51.160329: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 23:56:51.160369: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 23:56:51.160652: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpgeg4ll1i
2026-05-20 23:56:51.161544: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 23:56:51.161560: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpgeg4ll1i
2026-05-20 23:56:51.164137: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:382] MLIR V1 optimization pass is not enabled
2026-05-20 23:56:51.164661: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 23:56:51.188916: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpgeg4ll1i
2026-05

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)

In [14]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        # (b) Provide representative_data_gen(X_train_scaled).
        converter.representative_dataset = lambda: representative_data_gen(X_train_scaled)
        # (c) Set supported_ops to TFLITE_BUILTINS_INT8.
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        # (d) Set inference_input_type and inference_output_type to tf.int8.
        converter.inference_input_type  = tf.int8
        converter.inference_output_type = tf.int8

    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        # (b) Set supported_types to [tf.float16].
        converter.target_spec.supported_types = [tf.float16]

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations.
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.
    tflite_model = converter.convert()
    with open(filename, 'wb') as f:
        f.write(tflite_model)

    # Step 3: Run TFLite inference.
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model.
    # - Allocate tensors.
    # - Get input and output tensor details.
    # - If the input is quantized, quantize each test sample using scale and zero point.
    # - If the output is quantized, dequantize the prediction using scale and zero point.
    # - Collect predictions into y_pred using np.argmax.
    # - Compare with y_true = np.argmax(y_test_cat, axis=1).

    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()

    input_details  = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    in_scale,  in_zp  = input_details[0]['quantization']
    out_scale, out_zp = output_details[0]['quantization']

    y_true = np.argmax(y_test_cat, axis=1)
    y_pred = []

    for i in range(len(X_test)):
        sample = X_test[i:i + 1].astype(np.float32)

        # Quantize input if needed
        if input_details[0]['dtype'] == np.int8:
            if in_scale != 0:
                sample = np.round(sample / in_scale + in_zp).astype(np.int8)
            else:
                sample = sample.astype(np.int8)

        interpreter.set_tensor(input_details[0]['index'], sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details[0]['index']).copy()

        # Dequantize output if needed
        if output_details[0]['dtype'] == np.int8:
            if out_scale != 0:
                output = (output.astype(np.float32) - out_zp) * out_scale

        y_pred.append(np.argmax(output))

    y_pred = np.array(y_pred)

    # Step 4: Report results.
    print(f'\n{quant_type.upper()} TFLite model size: {file_size_kb(filename):.2f} KB')

    print('\nClassification Report:')
    print(classification_report(y_true, y_pred, target_names=['Class 0', 'Class 1', 'Class 2']))
    print('Confusion Matrix:')
    print(confusion_matrix(y_true, y_pred))

In [15]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'int8',    'model_int8.tflite')
quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'float16', 'model_float16.tflite')
quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'dynamic', 'model_dynamic.tflite')

INFO:tensorflow:Assets written to: /tmp/tmp2jwbbtc_/assets


INFO:tensorflow:Assets written to: /tmp/tmp2jwbbtc_/assets



INT8 TFLite model size: 5.74 KB

Classification Report:
              precision    recall  f1-score   support

     Class 0       0.95      1.00      0.97        19
     Class 1       1.00      0.95      0.98        21
     Class 2       1.00      1.00      1.00        14

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion Matrix:
[[19  0  0]
 [ 1 20  0]
 [ 0  0 14]]


/home/sam/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-20 23:56:51.505697: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 23:56:51.505721: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 23:56:51.505824: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp2jwbbtc_
2026-05-20 23:56:51.506499: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 23:56:51.506510: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmp2jwbbtc_
2026-05-20 23:56:51.508590: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 23:56:51.528019: I tensorflow/cc/saved_model/loader.c

INFO:tensorflow:Assets written to: /tmp/tmpvsnsoepy/assets


...

2026-05-20 23:56:51.809954: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 23:56:51.809980: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 23:56:51.810075: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpvsnsoepy
2026-05-20 23:56:51.810627: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 23:56:51.810638: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpvsnsoepy
2026-05-20 23:56:51.812164: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 23:56:51.831477: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpvsnsoepy
2026-05-20 23:56:51.836684: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 26609 m

INFO:tensorflow:Assets written to: /tmp/tmp_qx5wn7s/assets


INFO:tensorflow:Assets written to: /tmp/tmp_qx5wn7s/assets



DYNAMIC TFLite model size: 8.17 KB

Classification Report:
              precision    recall  f1-score   support

     Class 0       1.00      1.00      1.00        19
     Class 1       1.00      1.00      1.00        21
     Class 2       1.00      1.00      1.00        14

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion Matrix:
[[19  0  0]
 [ 0 21  0]
 [ 0  0 14]]


2026-05-20 23:56:52.241476: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 23:56:52.241501: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 23:56:52.241592: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmp_qx5wn7s
2026-05-20 23:56:52.242056: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 23:56:52.242063: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmp_qx5wn7s
2026-05-20 23:56:52.243442: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 23:56:52.261769: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmp_qx5wn7s
2026-05-20 23:56:52.266774: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 25183 m

## Problem 1 - Part (c)

### Pruning

In [16]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

batch_size_prune = 8
epochs_prune     = 10
dataset_size     = len(X_train_scaled)
end_step         = math.ceil(dataset_size / batch_size_prune) * epochs_prune

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=end_step
)

print(f"end_step: {end_step}")

end_step: 160


In [17]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

pruned_model = Sequential([
    prune_low_magnitude(
        Dense(64, activation='relu', input_shape=(num_features,)),
        pruning_schedule=pruning_schedule
    ),
    prune_low_magnitude(
        Dense(32, activation='relu'),
        pruning_schedule=pruning_schedule
    ),
    prune_low_magnitude(
        Dense(num_classes, activation='softmax'),
        pruning_schedule=pruning_schedule
    )
])

pruned_model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 prune_low_magnitude_dense_  (None, 64)                1730      
 3 (PruneLowMagnitude)                                           
                                                                 
 prune_low_magnitude_dense_  (None, 32)                4130      
 4 (PruneLowMagnitude)                                           
                                                                 
 prune_low_magnitude_dense_  (None, 3)                 197       
 5 (PruneLowMagnitude)                                           
                                                                 
Total params: 6057 (23.67 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 2982 (11.66 KB)
_________________________________________________________________


In [18]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

pruned_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [tfmot.sparsity.keras.UpdatePruningStep()]

pruned_history = pruned_model.fit(
    X_train_scaled, y_train_cat,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/10
13/13 [==============================] - 1s 8ms/step - loss: 0.9664 - accuracy: 0.5556 - val_loss: 0.8073 - val_accuracy: 0.8400
Epoch 2/10
13/13 [==============================] - 0s 2ms/step - loss: 0.6986 - accuracy: 0.8485 - val_loss: 0.6123 - val_accuracy: 0.8800
Epoch 3/10
13/13 [==============================] - 0s 2ms/step - loss: 0.5130 - accuracy: 0.8990 - val_loss: 0.4678 - val_accuracy: 0.8800
Epoch 4/10
13/13 [==============================] - 0s 2ms/step - loss: 0.3802 - accuracy: 0.9394 - val_loss: 0.3582 - val_accuracy: 0.9200
Epoch 5/10
13/13 [==============================] - 0s 2ms/step - loss: 0.2851 - accuracy: 0.9596 - val_loss: 0.2811 - val_accuracy: 0.9200
Epoch 6/10
13/13 [==============================] - 0s 2ms/step - loss: 0.2167 - accuracy: 0.9798 - val_loss: 0.2192 - val_accuracy: 0.9600
Epoch 7/10
13/13 [==============================] - 0s 2ms/step - loss: 0.1677 - accuracy: 0.9798 - val_loss: 0.1762 - val_accuracy: 0.9600
Epoch 8/10
13/13 [==

In [19]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

stripped_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

converter = tf.lite.TFLiteConverter.from_keras_model(stripped_model)
tflite_pruned = converter.convert()

with open('model_pruned.tflite', 'wb') as f:
    f.write(tflite_pruned)

print(f"Pruned TFLite model size: {file_size_kb('model_pruned.tflite'):.2f} KB")

INFO:tensorflow:Assets written to: /tmp/tmphqkiat73/assets


INFO:tensorflow:Assets written to: /tmp/tmphqkiat73/assets


Pruned TFLite model size: 14.14 KB


2026-05-20 23:56:53.857209: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 23:56:53.857231: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 23:56:53.857317: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmphqkiat73
2026-05-20 23:56:53.857617: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 23:56:53.857622: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmphqkiat73
2026-05-20 23:56:53.858296: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 23:56:53.867758: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmphqkiat73
2026-05-20 23:56:53.870745: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 13427 m

In [20]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

y_pred_probs_pruned = stripped_model.predict(X_test_scaled)
y_pred_pruned = np.argmax(y_pred_probs_pruned, axis=1)
y_true        = np.argmax(y_test_cat, axis=1)

print("Classification Report (Pruned Model):")
print(classification_report(y_true, y_pred_pruned, target_names=['Class 0', 'Class 1', 'Class 2']))

print("Confusion Matrix (Pruned Model):")
print(confusion_matrix(y_true, y_pred_pruned))

2/2 [==============================] - 0s 2ms/step
Classification Report (Pruned Model):
              precision    recall  f1-score   support

     Class 0       1.00      1.00      1.00        19
     Class 1       1.00      0.95      0.98        21
     Class 2       0.93      1.00      0.97        14

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion Matrix (Pruned Model):
[[19  0  0]
 [ 0 20  1]
 [ 0  0 14]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [21]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

student_model = Sequential([
    Dense(32, activation='relu', input_shape=(num_features,)),
    Dense(16, activation='relu'),
    Dense(num_classes, activation='softmax')
])

student_model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_6 (Dense)             (None, 32)                448       
                                                                 
 dense_7 (Dense)             (None, 16)                528       
                                                                 
 dense_8 (Dense)             (None, 3)                 51        
                                                                 
Total params: 1027 (4.01 KB)
Trainable params: 1027 (4.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [22]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

teacher_preds_soft = model.predict(X_train_scaled)
print("Teacher soft label shape:", teacher_preds_soft.shape)

4/4 [==============================] - 0s 1ms/step
Teacher soft label shape: (124, 3)


In [23]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

# (a) Concatenate hard and soft labels
y_train_combined = np.concatenate([y_train_cat, teacher_preds_soft], axis=1)
print("Combined label shape:", y_train_combined.shape)  # expected: (n_train, 6)


def distillation_loss(y_true_combined, y_pred):

    alpha = 0.5  # weight for hard vs. soft loss

    # Split the combined label into hard (ground truth) and soft (teacher) targets
    y_true_hard = y_true_combined[:, :num_classes]   # [:, :3]
    y_true_soft = y_true_combined[:, num_classes:]   # [:, 3:]

    # Compute hard-label loss
    hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)

    # Compute soft-label loss
    soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)

    # Combine with alpha weighting
    return alpha * hard_loss + (1.0 - alpha) * soft_loss

Combined label shape: (124, 6)


In [24]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

student_model.compile(
    optimizer='adam',
    loss=distillation_loss,
    metrics=['accuracy']
)

student_history = student_model.fit(
    X_train_scaled, y_train_combined,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

Epoch 1/10
13/13 [==============================] - 0s 8ms/step - loss: 1.2595 - accuracy: 0.3838 - val_loss: 1.1046 - val_accuracy: 0.3600
Epoch 2/10
13/13 [==============================] - 0s 2ms/step - loss: 1.0507 - accuracy: 0.5253 - val_loss: 0.9426 - val_accuracy: 0.5600
Epoch 3/10
13/13 [==============================] - 0s 2ms/step - loss: 0.8992 - accuracy: 0.5960 - val_loss: 0.8250 - val_accuracy: 0.6400
Epoch 4/10
13/13 [==============================] - 0s 2ms/step - loss: 0.7773 - accuracy: 0.6970 - val_loss: 0.7257 - val_accuracy: 0.8000
Epoch 5/10
13/13 [==============================] - 0s 2ms/step - loss: 0.6635 - accuracy: 0.8081 - val_loss: 0.6300 - val_accuracy: 0.8400
Epoch 6/10
13/13 [==============================] - 0s 2ms/step - loss: 0.5634 - accuracy: 0.8586 - val_loss: 0.5409 - val_accuracy: 0.8800
Epoch 7/10
13/13 [==============================] - 0s 2ms/step - loss: 0.4690 - accuracy: 0.9091 - val_loss: 0.4670 - val_accuracy: 0.9200
Epoch 8/10
13/13 [==

In [25]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.

converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
tflite_kd = converter.convert()

with open('model_kd.tflite', 'wb') as f:
    f.write(tflite_kd)

print(f"KD Student TFLite model size: {file_size_kb('model_kd.tflite'):.2f} KB")

INFO:tensorflow:Assets written to: /tmp/tmpipkoc72l/assets


INFO:tensorflow:Assets written to: /tmp/tmpipkoc72l/assets


KD Student TFLite model size: 6.10 KB


2026-05-20 23:56:54.914484: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 23:56:54.914509: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 23:56:54.914605: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpipkoc72l
2026-05-20 23:56:54.914989: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 23:56:54.914995: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpipkoc72l
2026-05-20 23:56:54.916178: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 23:56:54.933742: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /tmp/tmpipkoc72l
2026-05-20 23:56:54.938943: I tensorflow/cc/saved_model/loader.cc:316] SavedModel load for tags { serve }; Status: success: OK. Took 24338 m

In [26]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

y_pred_probs_kd = student_model.predict(X_test_scaled)
y_pred_kd = np.argmax(y_pred_probs_kd, axis=1)
y_true    = np.argmax(y_test_cat, axis=1)

print("Classification Report (KD Student Model):")
print(classification_report(y_true, y_pred_kd, target_names=['Class 0', 'Class 1', 'Class 2']))

print("Confusion Matrix (KD Student Model):")
print(confusion_matrix(y_true, y_pred_kd))

2/2 [==============================] - 0s 2ms/step
Classification Report (KD Student Model):
              precision    recall  f1-score   support

     Class 0       1.00      1.00      1.00        19
     Class 1       1.00      0.95      0.98        21
     Class 2       0.93      1.00      0.97        14

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion Matrix (KD Student Model):
[[19  0  0]
 [ 0 20  1]
 [ 0  0 14]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.

### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.

In [27]:
# Part (e): Further size reduction — KD student model + INT8 quantization
#
# Strategy: apply full INT8 quantization to the already-smaller student model.
# The student (32-16-3) is smaller than the teacher (64-32-3), and INT8 further
# reduces each weight from 32-bit float to 8-bit integer (~4x compression).

converter_e = tf.lite.TFLiteConverter.from_keras_model(student_model)
converter_e.optimizations = [tf.lite.Optimize.DEFAULT]
converter_e.representative_dataset = lambda: representative_data_gen(X_train_scaled)
converter_e.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter_e.inference_input_type  = tf.int8
converter_e.inference_output_type = tf.int8

tflite_kd_int8 = converter_e.convert()

with open('model_kd_int8.tflite', 'wb') as f:
    f.write(tflite_kd_int8)

print(f"KD Student + INT8 TFLite model size: {file_size_kb('model_kd_int8.tflite'):.2f} KB")

# Evaluate with TFLite interpreter
interpreter_e = tf.lite.Interpreter(model_path='model_kd_int8.tflite')
interpreter_e.allocate_tensors()

in_det_e  = interpreter_e.get_input_details()
out_det_e = interpreter_e.get_output_details()

in_scale_e,  in_zp_e  = in_det_e[0]['quantization']
out_scale_e, out_zp_e = out_det_e[0]['quantization']

y_true_e = np.argmax(y_test_cat, axis=1)
y_pred_e = []

for i in range(len(X_test_scaled)):
    sample = X_test_scaled[i:i + 1].astype(np.float32)
    if in_det_e[0]['dtype'] == np.int8:
        if in_scale_e != 0:
            sample = np.round(sample / in_scale_e + in_zp_e).astype(np.int8)
        else:
            sample = sample.astype(np.int8)
    interpreter_e.set_tensor(in_det_e[0]['index'], sample)
    interpreter_e.invoke()
    output = interpreter_e.get_tensor(out_det_e[0]['index']).copy()
    if out_det_e[0]['dtype'] == np.int8 and out_scale_e != 0:
        output = (output.astype(np.float32) - out_zp_e) * out_scale_e
    y_pred_e.append(np.argmax(output))

y_pred_e = np.array(y_pred_e)

print("\nClassification Report (KD Student + INT8):")
print(classification_report(y_true_e, y_pred_e, target_names=['Class 0', 'Class 1', 'Class 2']))
print("Confusion Matrix (KD Student + INT8):")
print(confusion_matrix(y_true_e, y_pred_e))

# Size comparison across all models
print("\n--- Model Size Comparison ---")
print(f"  Base float32:         {file_size_kb('model_base.tflite'):.2f} KB")
print(f"  Dynamic quantization: {file_size_kb('model_dynamic.tflite'):.2f} KB")
print(f"  Float16 quantization: {file_size_kb('model_float16.tflite'):.2f} KB")
print(f"  INT8 quantization:    {file_size_kb('model_int8.tflite'):.2f} KB")
print(f"  Pruned model:         {file_size_kb('model_pruned.tflite'):.2f} KB")
print(f"  KD student:           {file_size_kb('model_kd.tflite'):.2f} KB")
print(f"  KD student + INT8:    {file_size_kb('model_kd_int8.tflite'):.2f} KB")

INFO:tensorflow:Assets written to: /tmp/tmpe6bevo5l/assets


INFO:tensorflow:Assets written to: /tmp/tmpe6bevo5l/assets


KD Student + INT8 TFLite model size: 3.62 KB

Classification Report (KD Student + INT8):
              precision    recall  f1-score   support

     Class 0       1.00      1.00      1.00        19
     Class 1       1.00      0.95      0.98        21
     Class 2       0.93      1.00      0.97        14

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion Matrix (KD Student + INT8):
[[19  0  0]
 [ 0 20  1]
 [ 0  0 14]]

--- Model Size Comparison ---
  Base float32:         14.07 KB
  Dynamic quantization: 8.17 KB
  Float16 quantization: 8.95 KB
  INT8 quantization:    5.74 KB
  Pruned model:         14.14 KB
  KD student:           6.10 KB
  KD student + INT8:    3.62 KB


/home/sam/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-05-20 23:56:55.420265: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-05-20 23:56:55.420295: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-05-20 23:56:55.420421: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /tmp/tmpe6bevo5l
2026-05-20 23:56:55.421025: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-05-20 23:56:55.421036: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /tmp/tmpe6bevo5l
2026-05-20 23:56:55.422640: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-05-20 23:56:55.441181: I tensorflow/cc/saved_model/loader.c

# Problem 2: Exploring Edge Impulse (20 points)

### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.